# Dota 2 Stats Tracker — PySpark version
The same analysis as the SQL notebook, written with the Spark DataFrame API.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

MATCHES_TABLE = "workspace.default.matches"
HEROES_TABLE = "workspace.default.heroes"

matches = spark.table(MATCHES_TABLE)
heroes = spark.table(HEROES_TABLE)

## Prepare data: join heroes, add side, win and KDA columns

In [0]:
df = (
    matches
    .join(heroes, matches.hero_id == heroes.id, "left")
    .drop("id")
    .withColumn("is_radiant", F.col("player_slot") < 128)
    .withColumn("is_win", (F.col("is_radiant") == F.col("radiant_win").cast("boolean")).cast("int"))
    .withColumn("kda", (F.col("kills") + F.col("assists")) / F.greatest(F.col("deaths"), F.lit(1)))
    .withColumn("day", F.to_date(F.from_unixtime("start_time")))
)

display(df.select("match_id", "day", "localized_name", "kills", "deaths", "assists", "kda", "is_win")
          .orderBy(F.desc("start_time")))

match_id,day,localized_name,kills,deaths,assists,kda,is_win
9048444944,2026-10-17,Lion,3,12,16,1.5833333333333333,0
9048413977,2026-10-17,Lion,2,9,20,2.4444444444444446,1
9046946594,2026-10-16,Winter Wyvern,3,15,16,1.2666666666666666,0
9046923400,2026-10-16,Bane,2,9,5,0.7777777777777778,0
9046904626,2026-10-16,Bane,2,6,4,1.0,0
9046887347,2026-10-15,Bane,0,6,11,1.8333333333333333,1
9043908501,2026-10-13,Bane,5,6,20,4.166666666666667,1
9042496006,2026-10-13,Bane,6,8,26,4.0,0
9042414408,2026-10-12,Bane,1,12,11,1.0,0
9042380320,2026-10-12,Witch Doctor,6,8,17,2.875,1


## 1. General account statistics

In [0]:
account_stats = df.agg(
    F.count("*").alias("games"),
    F.round(100 * F.avg("is_win"), 2).alias("winrate"),
    F.round(F.avg("kills"), 2).alias("avg_kills"),
    F.round(F.avg("deaths"), 2).alias("avg_deaths"),
    F.round(F.avg("assists"), 2).alias("avg_assists"),
    F.round(F.avg("duration") / 60, 2).alias("avg_duration_min"),
)
display(account_stats)

games,winrate,avg_kills,avg_deaths,avg_assists,avg_duration_min
100,52.0,4.56,9.46,15.63,39.23


## 2. Statistics per side

In [0]:
side_stats = (
    df.withColumn("side", F.when(F.col("is_radiant"), "radiant").otherwise("dire"))
      .groupBy("side")
      .agg(
          F.count("*").alias("matches"),
          F.sum("is_win").alias("wins"),
          F.round(100 * F.avg("is_win"), 2).alias("winrate"),
      )
      .orderBy(F.desc("side"))
)
display(side_stats)

side,matches,wins,winrate
radiant,55,33,60.0
dire,45,19,42.22


## 3. Statistics per hero (min 3 games)

In [0]:
hero_stats = (
    df.groupBy("localized_name")
      .agg(
          F.count("*").alias("games"),
          F.sum("is_win").alias("wins"),
          F.round(100 * F.avg("is_win"), 2).alias("winrate"),
          F.round(F.avg("kills"), 2).alias("avg_kills"),
          F.round(F.avg("deaths"), 2).alias("avg_deaths"),
          F.round(F.avg("assists"), 2).alias("avg_assists"),
          F.round(F.avg("kda"), 2).alias("avg_kda"),
      )
      .filter(F.col("games") >= 3)          # same as HAVING COUNT(*) >= 3
      .withColumnRenamed("localized_name", "hero")
      .orderBy("hero")
)
display(hero_stats)

hero,games,wins,winrate,avg_kills,avg_deaths,avg_assists,avg_kda
Bane,49,33,67.35,3.8,9.69,18.49,2.88
Bounty Hunter,3,3,100.0,5.67,6.0,26.0,5.88
Death Prophet,3,0,0.0,7.67,9.67,11.33,1.97
Lion,5,3,60.0,2.8,9.0,12.2,1.87
Luna,4,2,50.0,7.75,5.5,10.5,3.48
Witch Doctor,10,5,50.0,5.2,8.8,12.7,2.57


## 4. Statistics per week (ISO week)

In [0]:
weekly = (
    df.withColumn(
          "week",
          F.concat(
              F.expr("extract(YEAROFWEEK FROM day)").cast("string"),
              F.lit("-"),
              F.lpad(F.weekofyear("day").cast("string"), 2, "0"),
          ),
      )
      .groupBy("week")
      .agg(
          F.count("*").alias("games"),
          F.round(100 * F.avg("is_win"), 2).alias("winrate"),
          F.round(F.avg("kda"), 2).alias("avg_kda"),
      )
      .orderBy("week")
)
display(weekly)

week,games,winrate,avg_kda
2026-26,8,50.0,1.9
2026-27,7,14.29,1.09
2026-28,2,0.0,1.69
2026-29,4,50.0,2.27
2026-30,6,66.67,3.68
2026-31,2,0.0,1.73
2026-32,7,57.14,2.75
2026-33,5,20.0,1.98
2026-34,2,100.0,2.25
2026-36,15,66.67,3.66


## 5. Cumulative winrate (window function)

In [0]:
w = Window.orderBy("start_time").rowsBetween(Window.unboundedPreceding, Window.currentRow)

cumulative = (
    df.withColumn("wins_so_far", F.sum("is_win").over(w))
      .withColumn("games_so_far", F.count("*").over(w))
      .withColumn("cumulative_winrate", F.round(100 * F.col("wins_so_far") / F.col("games_so_far"), 2))
      .select("day", "is_win", "wins_so_far", "games_so_far", "cumulative_winrate")
      .orderBy(F.desc("games_so_far"))
)
display(cumulative)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


day,is_win,wins_so_far,games_so_far,cumulative_winrate
2026-10-17,0,52,100,52.0
2026-10-17,1,52,99,52.53
2026-10-16,0,51,98,52.04
2026-10-16,0,51,97,52.58
2026-10-16,0,51,96,53.13
2026-10-15,1,51,95,53.68
2026-10-13,1,50,94,53.19
2026-10-13,0,49,93,52.69
2026-10-12,0,49,92,53.26
2026-10-12,1,49,91,53.85


## 6. Cross-check: PySpark results must match the SQL notebook

In [0]:
sql_check = spark.sql(f"""
    SELECT ROUND(100 * AVG(CAST((player_slot < 128) = CAST(radiant_win AS BOOLEAN) AS INT)), 2) AS winrate
    FROM {MATCHES_TABLE}
""").first()["winrate"]

pyspark_check = account_stats.first()["winrate"]

print(f"Winrate SQL: {sql_check}, PySpark: {pyspark_check}, match: {sql_check == pyspark_check}")

Winrate SQL: 52.0, PySpark: 52.0, match: True
